<a href="https://colab.research.google.com/github/dbonacorsi/AML-Basic/blob/main/AMLBas_2_DataPrep_ExploreData_part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Explore your data with descriptive statistics

***The time spent in understanding your data (BEFORE APPLYING ANY ML TECHNIQUES) is time well spent***.

In this notebook we start exploring **EDA** (**exploratory data analysis**): the process of analyzing and understanding your dataset before building models.

Suggestions and guidelines may differ, but here is what we use as a checklist:

1. Look at the raw data
1. Dimensions
1. Data types
1. Descriptive statistics (incl. missing values)
1. Distribution of instances across classes
1. Correlations
1. Skews

## 1. Raw data

In [ ]:
import pandas as pd

url = 'https://raw.githubusercontent.com/dbonacorsi/AML-Basic/refs/heads/main/datasets/pima-indians-diabetes.data.csv'

names = ['preg', 'plas', 'pres', 'skin', 'test', 'mass', 'pedi', 'age', 'class']
data = pd.read_csv(url, names=names)
data

More details can be found e.g. at https://www.kaggle.com/uciml/pima-indians-diabetes-database/data, and are summarized below:

    Pregnancies = Number of times pregnant
    Glucose = Plasma glucose concentration a 2 hours in an oral glucose tolerance test numeric
    BloodPressure = Diastolic blood pressure (mm Hg)
    SkinThickness = Triceps skin fold thickness (mm)
    Insulin = 2-Hour serum insulin (mu U/ml)
    BMI = Body mass index (weight in kg/(height in m)^2)
    DiabetesPedigreeFunction = Diabetes pedigree function
    Age = Age (years)
    Outcome = Class variable (0 or 1)

This is the mapping to actual code short names, in the same order:

    'preg'
    'plas'
    'pres'
    'skin'
    'test'
    'mass'
    'pedi'
    'age'
    'class'

## 2. Dimensions

In [ ]:
shape = data.shape
shape

## 3. Data types

In [ ]:
types = data.dtypes
print(types)

## 4. Descriptive statistics

Help on pandas set_option [here](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.set_option.html).


In [ ]:
from pandas import set_option

set_option('display.width', 200)
set_option('display.max_rows', 500)
set_option('display.max_columns', 500)
set_option('display.precision', 3)

In [ ]:
description = data.describe()
print(description)

What can we learn from this simple overview of our data?
1.   Are there missing entries anywhere? Anything suspicious? (*)
2.   Are the values of the features on the same scales? (**)



(*) e.g. can you explain the fact that the describe() output shows min = 0 for features that cannot be 0 physiologically?


(**) not at all.. more on this in one of next notebooks!

In [ ]:
# Features where 0 is physiologically impossible: pandas reads them as 0, not as NaN
cols = ['plas', 'pres', 'skin', 'test', 'mass']

# Count the 0 placeholders per column
print("Zeros per column (likely missing values):")
print((data[cols] == 0).sum())

# Count real NaN values (expected to be 0 everywhere, since the file has no blank fields)
print("\nNaN values per column:")
print(data.isnull().sum())

Let's discuss this.

**Should I remove the features?**. No, at least not yet.

Why? Dropping a whole feature throws away the other 767 valid values in it, and most of the zeros are only a few percent of the rows in three of the five columns. The usual approach is to treat the zeros as missing, then decide how to fill them.

Your options, roughly in order of preference:

*   *Replace 0 with NaN, then impute (recommended)*. Fill each gap with the column median, which is less sensitive to outliers than the mean. Do this inside a pipeline, fitted on the training split only, so the test set doesn’t leak into the fill values.

*   *Drop the rows with missing values*. Simple, but test (374 zeros) would remove about half the dataset, so it’s only reasonable for small losses.

*   *Drop the feature*. This is defensible for test (about half its values are missing) and possibly skin (227 of 768). Decide it on evidence: compare model accuracy with and without it, rather than by eye.

*   *Add a “was missing” indicator column* (for example test_missing), so the model can learn from the fact that the value was absent. Consider treat this as a one-hot-encoded info?

At the moment we are just exploring and familiarizing, exploiting this (very educational) dataset. But it this would have been a real-world dataset and a real ML project, this aspect would have been a point to seriously address!


## 5. Class distribution
(only for classification problems)

Just counting..

In [ ]:
class_counts = data.groupby('class').size()
print(class_counts)

Or checking proportions..

In [ ]:
class_proportions = data['class'].value_counts(normalize=True)
print(class_proportions.round(3))

The split is about 65/35, which is a mild imbalance: it is worth mentioning that - in this mild case already - accuracy alone might be misleading.. be carefull (if this would be a real project!)

## 6. Correlations between attributes

In [ ]:
set_option('display.width', 100)
#set_option('precision', 3)

correlations = data.corr(method='pearson')
correlations

Show which features correlate most with the target (NOTE: Pearson correlation only captures linear relationships, though..)

In [ ]:
correlations['class'].sort_values(ascending=False)

## 7. Skew of Univariate Distributions

In [ ]:
skew = data.skew()
print(skew)

Rule of thumb: e.g. `|skew| > 1` means strongly skewed.


## Summary

What we did:

* we familiarized with some quick ways to have a glance at my dataset and try to describe and summarize it statistically
* we implemented these concepts by exploiting Pandas' functionalities: easy, basically one command each.
* remember to look carefully at the numbers, ask yourself questions as of whether things are as you see them, and note down everything that comes to mind as it might well be useful at a later stage!

## What's next

We did what we did with numbers. ***Isn't it frustrating?*** Time to make it with visualizations!